# Aggregation Level and Match Analysis: Data Processing

This notebook demonstrates how the **aggregation level** at which football (soccer) event data are analysed changes the estimated associations between performance indicators (PIs) and match success — a form of *noncollapsibility* described in the accompanying manuscript.

## Data

**StatsBomb open data** — 1. Bundesliga 2015/16 season, 306 matches, ≈ 600 000 events (loaded via [statsbombpy](https://github.com/statsbomb/statsbombpy)).

## Four Aggregation Levels

| # | Level | Unit of analysis | N (approx.) | Success criterion |
|---|-------|-----------------|-------------|-------------------|
| 1 | Season | Team × season | 18 rows | Rank category (relegated → Champions League) |
| 2 | Match | Team × match | 612 rows | Match result (away win / draw / home win) |
| 3 | Scoreline | Team × scoreline segment | ~2 000 rows | Next goal scorer |
| 4 | Possession | Team × possession sequence | ~60 000 rows | Next goal scorer |

Each level fits an **ordered probit** model with a standardised PI difference as the sole predictor (plus match-status covariates at the scoreline and possession levels).


In [1]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

from statsbombpy import sb

## Performance Indicators

The following 17 PIs are extracted from event data. At each aggregation level, the *difference* between the home and away team (or team and opponent) is computed and z-standardised before entering the model.


In [2]:
cols = [
    'match_id', 'period', 'minute', 'second', 'timestamp', 'id', 'index', 'type', 'player', 'team', 'play_pattern', 'possession', 'possession_team',
    'under_pressure', 'tactics', 'duration', 'player_id', 'position', 'possession_team_id', 'team_id',
    'home_team', 'away_team', 'home_goals_ft', 'away_goals_ft', 'match_week',
    'dribble_outcome', 'duel_outcome', 'duel_type', 'location', 'pass_cross', 'pass_end_location', 
    'pass_height', 'pass_length', 'pass_outcome',  'pass_angle', 'pass_through_ball', 'pass_type', 
    'shot_outcome', 'shot_statsbomb_xg', 'shot_type'
]
performance_indicators = [
    'xg', 'shots',
    'passes', 'highpasses', 'groundpasses', 'pressuredpasses', 'pass_length',
    'crosses', 'dribbles', 
    'tackles', 'tackles_success', 'interceptions', 'pressures', 'blocks', 'fouls',
    'counterattacks'
]

## Four Aggregation Levels — Overview

| Level | Unit | N (approx.) | Success criterion | Notes |
|-------|------|-------------|-------------------|-------|
| Season | Team × season | 18 | Rank category (0 = relegated, 3 = CL) | PIs summed over all 34 match-days |
| Match | Team × match | 612 | Match result (0 = away win, 1 = draw, 2 = home win) | PIs summed for full match |
| Scoreline | Team × scoreline segment | ~2 000 | Next goal (0 = away, 1 = none, 2 = home) | Normalised per-90 min; segments > 60 s only |
| Possession | Team × possession | ~60 000 | Next goal (0 = away, 1 = none, 2 = home) | Normalised per-90 min; possessions > 10 s only |

At the scoreline and possession levels, **match-status** covariates (home leading / away leading vs draw) are added to control for the systematic changes in team behaviour driven by the current score.


## Helper Functions


### Process match data

In [3]:
def add_match_info(df):

# sort events by match, period, and timestamp
    df = df.sort_values(['period', 'timestamp']).reset_index(drop=True)

# increment each time the scoreline changes
    df['scoreline_id'] = df['scoreline'].ne(df['scoreline'].shift()).cumsum()

### Create Next Goal column
    scoreline_by_group = df.groupby('scoreline_id')['scoreline'].first()
    scoreline_change = scoreline_by_group.shift(-1) - scoreline_by_group

# Create a DataFrame showing if the next scoreline is higher, lower, or the same
    scoreline_trend = pd.DataFrame({
        'current_scoreline': scoreline_by_group,
        'next_scoreline': scoreline_by_group.shift(-1),
        'next_goal': np.where(scoreline_change > 0, 'home',
                np.where(scoreline_change < 0, 'away', 'none'))
    })

    df = pd.merge(
        df,
        scoreline_trend.reset_index()[['scoreline_id', 'next_goal']],
        on='scoreline_id',
        how='left'
    )
    
### Create absolute timestamp in seconds
    secs_1st_half = pd.to_timedelta(
        df[df.period == 1]['timestamp']).dt.total_seconds().max()

    df['absolute_sec'] = np.where(
        df['period'] == 1,
        pd.to_timedelta(df['timestamp']).dt.total_seconds(),
        pd.to_timedelta(df['timestamp']).dt.total_seconds() + secs_1st_half
    )

    return df


`data_aggregation()` dispatches to one of three aggregation routines:

- **`match_level_agg()`** — one row per team × match
- **`scoreline_agg()`** — one row per constant-scoreline segment
- **`possession_agg()`** — one row per StatsBomb possession sequence


In [4]:
grouping_cols_scoreline = [
    'match_id', 'scoreline_id', 'home_score', 'away_score', 'scoreline', 'match_status', 'next_goal',
    'home_team', 'away_team', 'home_goals_ft', 'away_goals_ft', 'match_outcome', 'match_week'
    ]

def scoreline_agg(df):
    return pd.Series({
        'duration_s': df['absolute_sec'].max() - df['absolute_sec'].min(),
        'home_xg': df.loc[df['location'] == 'home', 'shot_statsbomb_xg'].sum(),
        'away_xg': df.loc[df['location'] == 'away', 'shot_statsbomb_xg'].sum(),
        'home_passes': ((df['type'] == 'Pass') & (df['location'] == 'home')).sum(),
        'away_passes': ((df['type'] == 'Pass') & (df['location'] == 'away')).sum(),
        'home_highpasses': ((df['pass_height'] == 'High Pass') & (df['location'] == 'home')).sum(),
        'away_highpasses': ((df['pass_height'] == 'High Pass') & (df['location'] == 'away')).sum(),
        'home_groundpasses': ((df['pass_height'] == 'Ground Pass') & (df['location'] == 'home')).sum(),
        'away_groundpasses': ((df['pass_height'] == 'Ground Pass') & (df['location'] == 'away')).sum(),
        'home_pressuredpasses': ((df['pressured_pass'] == True) & (df['location'] == 'home')).sum(),
        'away_pressuredpasses': ((df['pressured_pass'] == True) & (df['location'] == 'away')).sum(),
        'home_crosses': ((df['pass_cross'] == True) & (df['location'] == 'home')).sum(),
        'away_crosses': ((df['pass_cross'] == True) & (df['location'] == 'away')).sum(),
        'home_dribbles': ((df['type'] == 'Dribble') & (df['location'] == 'home')).sum(),
        'away_dribbles': ((df['type'] == 'Dribble') & (df['location'] == 'away')).sum(),
        'home_shots': ((df['type'] == 'Shot') & (df['location'] == 'home')).sum(),
        'away_shots': ((df['type'] == 'Shot') & (df['location'] == 'away')).sum(),
        'home_pass_length': df.loc[df['location'] == 'home', 'pass_length'].sum(),
        'away_pass_length': df.loc[df['location'] == 'away', 'pass_length'].sum(),
        'home_tackles': ((df['duel_type'] == 'Tackle') & (df['location'] == 'home')).sum(),
        'away_tackles': ((df['duel_type'] == 'Tackle') & (df['location'] == 'away')).sum(),
        'home_tackles_success': ((df['duel_type'] == 'Tackle') & (df['location'] == 'home') & (df['duel_outcome'].isin(['Won', 'Success', 'Success In Play', 'Success Out']))).sum(),
        'away_tackles_success': ((df['duel_type'] == 'Tackle') & (df['location'] == 'away') & (df['duel_outcome'].isin(['Won', 'Success', 'Success In Play', 'Success Out']))).sum(),
        'home_interceptions': ((df['type'] == 'Interception') & (df['location'] == 'home')).sum(),
        'away_interceptions': ((df['type'] == 'Interception') & (df['location'] == 'away')).sum(),
        'home_pressures': ((df['type'] == 'Pressure') & (df['location'] == 'home')).sum(),
        'away_pressures': ((df['type'] == 'Pressure') & (df['location'] == 'away')).sum(),
        'home_blocks': ((df['type'] == 'Block') & (df['location'] == 'home')).sum(),
        'away_blocks': ((df['type'] == 'Block') & (df['location'] == 'away')).sum(),
        'home_fouls': ((df['type'] == 'Foul Committed') & (df['location'] == 'home')).sum(),
        'away_fouls': ((df['type'] == 'Foul Committed') & (df['location'] == 'away')).sum(),
        'home_counterattacks': ((df['play_pattern'] == 'From Counter') & (df['location'] == 'home')).sum(),
        'away_counterattacks': ((df['play_pattern'] == 'From Counter') & (df['location'] == 'away')).sum(),
    })

grouping_cols_possession = [
    'match_id', 'possession', 'home_score', 'away_score', 'scoreline', 'match_status', 'possession_team', 'next_goal',
    'home_team', 'away_team', 'home_goals_ft', 'away_goals_ft', 'match_outcome', 'match_week'
]

def possession_agg(df):
    return pd.Series({
        'duration_s': df['absolute_sec'].max() - df['absolute_sec'].min(),
        'n_events': df.shape[0],
        'home_goals': df.loc[df['location'] == 'home', 'home_goal'].sum(),
        'away_goals': df.loc[df['location'] == 'away', 'away_goal'].sum(),
        'home_xg': df.loc[df['location'] == 'home', 'shot_statsbomb_xg'].sum(),
        'away_xg': df.loc[df['location'] == 'away', 'shot_statsbomb_xg'].sum(),
        'home_passes': ((df['type'] == 'Pass') & (df['location'] == 'home')).sum(),
        'away_passes': ((df['type'] == 'Pass') & (df['location'] == 'away')).sum(),
        'home_highpasses': ((df['pass_height'] == 'High Pass') & (df['location'] == 'home')).sum(),
        'away_highpasses': ((df['pass_height'] == 'High Pass') & (df['location'] == 'away')).sum(),
        'home_groundpasses': ((df['pass_height'] == 'Ground Pass') & (df['location'] == 'home')).sum(),
        'away_groundpasses': ((df['pass_height'] == 'Ground Pass') & (df['location'] == 'away')).sum(),
        'home_pressuredpasses': ((df['pressured_pass'] == True) & (df['location'] == 'home')).sum(),
        'away_pressuredpasses': ((df['pressured_pass'] == True) & (df['location'] == 'away')).sum(),
        'home_crosses': ((df['pass_cross'] == True) & (df['location'] == 'home')).sum(),
        'away_crosses': ((df['pass_cross'] == True) & (df['location'] == 'away')).sum(),
        'home_dribbles': ((df['type'] == 'Dribble') & (df['location'] == 'home')).sum(),
        'away_dribbles': ((df['type'] == 'Dribble') & (df['location'] == 'away')).sum(),
        'home_shots': ((df['type'] == 'Shot') & (df['location'] == 'home')).sum(),
        'away_shots': ((df['type'] == 'Shot') & (df['location'] == 'away')).sum(),
        'home_pass_length': df.loc[df['location'] == 'home', 'pass_length'].sum(),
        'away_pass_length': df.loc[df['location'] == 'away', 'pass_length'].sum(),
        'home_tackles': ((df['duel_type'] == 'Tackle') & (df['location'] == 'home')).sum(),
        'away_tackles': ((df['duel_type'] == 'Tackle') & (df['location'] == 'away')).sum(),
        'home_tackles_success': ((df['duel_type'] == 'Tackle') & (df['location'] == 'home') & (df['duel_outcome'].isin(['Won', 'Success', 'Success In Play', 'Success Out']))).sum(),
        'away_tackles_success': ((df['duel_type'] == 'Tackle') & (df['location'] == 'away') & (df['duel_outcome'].isin(['Won', 'Success', 'Success In Play', 'Success Out']))).sum(),
        'home_interceptions': ((df['type'] == 'Interception') & (df['location'] == 'home')).sum(),
        'away_interceptions': ((df['type'] == 'Interception') & (df['location'] == 'away')).sum(),
        'home_pressures': ((df['type'] == 'Pressure') & (df['location'] == 'home')).sum(),
        'away_pressures': ((df['type'] == 'Pressure') & (df['location'] == 'away')).sum(),
        'home_blocks': ((df['type'] == 'Block') & (df['location'] == 'home')).sum(),
        'away_blocks': ((df['type'] == 'Block') & (df['location'] == 'away')).sum(),
        'home_fouls': ((df['type'] == 'Foul Committed') & (df['location'] == 'home')).sum(),
        'away_fouls': ((df['type'] == 'Foul Committed') & (df['location'] == 'away')).sum(),
        'home_counterattacks': ((df['play_pattern'] == 'From Counter') & (df['location'] == 'home')).sum(),
        'away_counterattacks': ((df['play_pattern'] == 'From Counter') & (df['location'] == 'away')).sum(),
        'att_goals': df.loc[df['team_role'] == 'attacking', 'Goal'].sum(),
        'att_xg': df.loc[df['team_role'] == 'attacking', 'shot_statsbomb_xg'].sum(),
        'att_shots': df.loc[df['team_role'] == 'attacking', 'type'].eq('Shot').sum(),
        'att_passes': df.loc[df['team_role'] == 'attacking', 'type'].eq('Pass').sum(),
        'att_highpasses': df.loc[df['team_role'] == 'attacking', 'pass_height'].eq('High Pass').sum(),
        'att_groundpasses': df.loc[df['team_role'] == 'attacking', 'pass_height'].eq('Ground Pass').sum(),
        'att_crosses': df.loc[df['team_role'] == 'attacking', 'pass_cross'].eq(True).sum(),
        'att_dribbles': df.loc[df['team_role'] == 'attacking', 'type'].eq('Dribble').sum(),
        'att_pass_length': df.loc[df['team_role'] == 'attacking', 'pass_length'].sum(),
        'def_tackles': ((df['duel_type'] == 'Tackle') & (df['team_role'] == 'defending')).sum(),
        'def_tackles_success': ((df['duel_type'] == 'Tackle') & (df['team_role'] == 'defending') & (df['duel_outcome'].isin(['Won', 'Success', 'Success In Play', 'Success Out']))).sum(),
        'def_interceptions': ((df['type'] == 'Interception') & (df['team_role'] == 'defending')).sum(),
        'def_pressures': ((df['type'] == 'Pressure') & (df['team_role'] == 'defending')).sum(),
        'def_blocks': ((df['type'] == 'Block') & (df['team_role'] == 'defending')).sum(),
        'def_fouls': ((df['type'] == 'Foul Committed') & (df['team_role'] == 'defending')).sum()
    })

def match_level_agg(df):
    df_matches_long = df.groupby('team').agg(
        match_id=('match_id', 'first'),
        location=('location', 'first'),
        match_outcome=('match_outcome', 'first'),
        goals=('Goal', 'sum'),
        xg=('shot_statsbomb_xg', 'sum'),
        shots=('type', lambda x: (x == 'Shot').sum()),
        passes=('type', lambda x: (x == 'Pass').sum()),
        pressuredpasses=('pressured_pass', 'sum'),
        pressureddribbles=('pressured_dribble', 'sum'),
        highpasses=('pass_height', lambda x: (x == 'High Pass').sum()),
        groundpasses=('pass_height', lambda x: (x == 'Ground Pass').sum()),
        pass_length=('pass_length', 'sum'),
        crosses=('pass_cross', lambda x: (x == True).sum()),
        dribbles=('type', lambda x: (x == 'Dribble').sum()),
        tackles=('duel_type', lambda x: (x == 'Tackle').sum()),
        tackles_success=('duel_outcome', lambda x: x.isin(['Won', 'Success', 'Success In Play', 'Success Out']).sum()),
        interceptions=('type', lambda x: (x == 'Interception').sum()),
        pressures=('type', lambda x: (x == 'Pressure').sum()),
        blocks=('type', lambda x: (x == 'Block').sum()),
        fouls=('type', lambda x: (x == 'Foul Committed').sum()),
        counterattacks=('play_pattern', lambda x: (x == 'From Counter').sum()),
    ).reset_index()

    m_id = df_matches_long['match_id'].unique()[0]
    match_outcome = df_matches_long['match_outcome'].unique()[0]

    ls_loc_dfs = []
    for loc in ['home', 'away']:
        df_temp = df_matches_long.loc[df_matches_long['location'] == loc].reset_index(drop=True).copy()
        df_temp = df_temp.drop(columns=['location', 'match_id', 'match_outcome'])
        df_temp = df_temp.add_prefix(f'{loc}_', axis=1)
        ls_loc_dfs.append(df_temp)

    df_matches_wide = pd.concat(ls_loc_dfs, axis=1)
    df_matches_wide.insert(0, 'match_id', m_id)
    df_matches_wide.insert(1, 'match_outcome', match_outcome)
    return df_matches_wide

def data_aggregation(df_match, level='match'):
    if level == 'match':
        return match_level_agg(df_match)
    elif level == 'scoreline':
        return df_match.groupby(grouping_cols_scoreline).apply(scoreline_agg).reset_index()
    elif level == 'possession':
        return df_match.groupby(grouping_cols_possession).apply(possession_agg).reset_index()
    else:
        raise ValueError("Invalid level specified. Use 'match', 'scoreline', or 'possession'.")


## Loading StatsBomb Open Data

[StatsBombPy](https://github.com/statsbomb/statsbombpy) provides free event-level data for selected competitions. Each **event** is a single action (pass, shot, pressure, …) with timestamp, team, player, and outcome attributes.

We use the **1. Bundesliga 2015/16** season (`competition_id = 9`, `season_id = 27`): 306 matches, 18 teams, ≈ 600 000 events.


In [5]:
sb.competitions()[sb.competitions()['competition_name'] == '1. Bundesliga']

/home/max/drive/projects/6_context/paper/.venv/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
0,9,281,Germany,1. Bundesliga,male,False,False,2023/2024,2024-09-28T20:46:38.893391,2025-07-06T04:26:07.636270,2025-07-06T04:26:07.636270,2024-09-28T20:46:38.893391
1,9,27,Germany,1. Bundesliga,male,False,False,2015/2016,2024-05-19T11:11:14.192381,NaN,NaN,2024-05-19T11:11:14.192381


### Load match metadata


In [6]:
matches_bl1516 = sb.matches(competition_id=9, season_id=27)


/home/max/drive/projects/6_context/paper/.venv/lib/python3.12/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


### Load all events for the season


In [ ]:
events = sb.competition_events(
    country="Germany",
    division= "1. Bundesliga",
    season="2015/2016",
    gender="male"
)

match_stats = matches_bl1516[['match_id', 'home_team', 'away_team', 'home_score', 'away_score', 'match_week']]
match_stats.columns = ['match_id', 'home_team', 'away_team', 'home_goals_ft', 'away_goals_ft', 'match_week']


events = events.merge(
    match_stats,
    on='match_id',
    how='left'
)

events = events[cols]

## Data Preprocessing

Starting from the raw event stream, we create the following derived columns:

- **`Goal`** — flag for shots resulting in a goal or own-goal events
- **`location`** — `"home"` or `"away"` for the team performing the action
- **`team_role`** — `"attacking"` (possession team) or `"defending"` (non-possession team)
- **`home_score` / `away_score`** — cumulative goals at each event
- **`scoreline`** — signed goal difference (`home_score − away_score`)
- **`match_status`** — `"home_leading"`, `"draw"`, or `"away_leading"`
- **`match_outcome`** — full-time result (`"home_win"`, `"draw"`, `"away_win"`)
- **`pressured_pass`** / **`pressured_dribble`** — actions performed under pressure


In [ ]:
# Create a Goal flag
events['Goal'] = (events['shot_outcome'].eq('Goal')) | (events['type'].eq('Own Goal For'))

# assign whether team executing the action is home or away
events['location'] = np.where(
    events['team'].eq(events['home_team']),
    'home', 'away'
)

# assign whether team executing the action is the possession team or not
events['team_role'] = np.where(
    events['possession_team'] == events['team'], 'attacking', 'defending'
)

events['home_goal'] = events['Goal'] & events['location'].eq('home')
events['away_goal'] = events['Goal'] & events['location'].eq('away')

# Ensure proper chronological order within each match
events = events.sort_values(['match_id', 'period', 'timestamp'])

# Running scores within each match
events['home_score'] = (
    events.groupby('match_id')['home_goal'].cumsum().astype(int)
)
events['away_score'] = (
    events.groupby('match_id')['away_goal'].cumsum().astype(int)
)

# Scoreline and match status
events['scoreline'] = events['home_score'] - events['away_score']
events['match_status'] = np.where(
    events['scoreline'].eq(0), 'draw',
    np.where(events['scoreline'] < 0, 'away_leading', 'home_leading')
)

# Discrete match outcome 
events['match_outcome'] = np.where(
    events['home_goals_ft'] > events['away_goals_ft'], 'home_win',
    np.where(events['home_goals_ft'] < events['away_goals_ft'], 'away_win', 'draw')
)

# Create pressured pass and pressured dribble columns
events['pressured_pass'] = (events['type'] == 'Pass') & (events['under_pressure'] == True)
events['pressured_dribble'] = (events['type'] == 'Dribble') & (events['under_pressure'] == True)

## Split Data into Per-Match Lists

Each match is extracted into its own DataFrame. Aggregation functions are applied per match to avoid cross-match contamination.


In [ ]:
ls_dfs_events = [
    events.loc[events['match_id'] == m].copy() for m in events['match_id'].unique()
]

#df_match = ls_dfs_events[3].copy()

## Aggregating Data at Four Levels

The loop below calls `data_aggregation()` at match, scoreline, and possession levels for every match. Season-level aggregation is handled separately by summing match-level differentials.


In [ ]:
ls_dfs_matchlevel = []
ls_dfs_scorelinelevel = []
ls_dfs_possessionlevel = []

for df in tqdm((ls_dfs_events), total=len(ls_dfs_events)):
    df = add_match_info(df)
    df_match = data_aggregation(df, level='match')
    ls_dfs_matchlevel.append(df_match)

    df_scoreline = data_aggregation(df, level='scoreline')
    ls_dfs_scorelinelevel.append(df_scoreline)

    df_possession = data_aggregation(df, level='possession')
    ls_dfs_possessionlevel.append(df_possession)

df_matchlevel = pd.concat(ls_dfs_matchlevel, ignore_index=True)
df_scorelinelevel = pd.concat(ls_dfs_scorelinelevel, ignore_index=True)
df_possessionlevel = pd.concat(ls_dfs_possessionlevel, ignore_index=True)
    

  0%|          | 0/306 [00:00<?, ?it/s]

### Season Level

Season-level data are derived from match-level data by:

1. Computing each team’s PI *differential* (team minus opponent) per match
2. Flipping signs for away-team rows so a positive value always means “outperformed the opponent”
3. Summing all 34 match-day differentials per team

The success criterion is **`rank_category`**: Champions League (ranks 1–4), Europa League (5–7), mid-table (8–15), and relegation (16–18), mapped to ordered integers 3–0.


In [ ]:
for pi in performance_indicators + ['goals']:
    df_matchlevel[pi + '_diff'] = df_matchlevel['home_' + pi] - df_matchlevel['away_' + pi]


In [ ]:
df_home = df_matchlevel[['match_id', 'home_team'] + [pi + '_diff' for pi in performance_indicators + ['goals']]]
df_home['location'] = 'home'
df_home.columns = df_home.columns.str.replace('home_', '')

df_away = df_matchlevel[['match_id', 'away_team'] + [pi + '_diff' for pi in performance_indicators + ['goals']]]
df_away[[pi + '_diff' for pi in performance_indicators + ['goals']]] = df_away[[pi + '_diff' for pi in performance_indicators + ['goals']]] * (-1)
df_away['location'] = 'away'
df_away.columns = df_away.columns.str.replace('away_', '')

df_season_temp = pd.concat([df_home, df_away], ignore_index=True)
df_season_temp['points'] = (df_season_temp.goals_diff > 0) * 3 + (df_season_temp.goals_diff == 0) * 1

In [ ]:
df_season =pd.concat([
    df_season_temp.groupby('team')[[pi + '_diff' for pi in performance_indicators + ['goals']]].sum(),
    df_season_temp.groupby('team')['points'].sum()
], axis=1).sort_values('points', ascending=False)

df_season['rank'] = range(1, len(df_season) + 1)

df_season['rank_category'] = pd.cut(
    df_season['rank'],
    bins=[0, 5, 8, 16, 19],
    labels=['CL', 'EL', 'middle', 'relegation'],
    right=False
)

### Per-90 Normalisation

Scoreline segments and possession sequences vary greatly in duration. We normalise each PI count by dividing by the segment duration (in minutes) and scaling to 90 minutes:

$$\text{PI}_{90} = \frac{\text{PI\_count}}{\text{duration (min)}} \times 90$$

Match-level data use raw counts (a full match is a fixed ~90-minute unit) and are **not** normalised.


In [ ]:
new_cols_possession = {}
new_cols_scoreline = {}
for pi in performance_indicators:
    for loc in ['home_', 'away_']:
        col = loc + pi
        if col in df_possessionlevel.columns:
            new_cols_possession[col + '_90'] = df_possessionlevel[col] / (df_possessionlevel['duration_s'] / 60) * 90
        if col in df_scorelinelevel.columns:
            new_cols_scoreline[col + '_90'] = df_scorelinelevel[col] / (df_scorelinelevel['duration_s'] / 60) * 90

df_possessionlevel = df_possessionlevel.assign(**new_cols_possession)
df_scorelinelevel = df_scorelinelevel.assign(**new_cols_scoreline)


### Exclude scoreline segments that lasted less than a minute

In [ ]:
df_scorelinelevel = df_scorelinelevel.loc[df_scorelinelevel['duration_s'] > 60]
df_possessionlevel = df_possessionlevel.loc[df_possessionlevel['duration_s'] > 10]

### Scaling

Some PIs span very different numeric ranges — `pass_length` is in metres (hundreds per game) while `xg` is in probability units (0–1). Each PI is divided by a constant scaling factor to bring values into comparable ranges for visual comparison.

This scaling is cosmetic: `pi_diff` is re-standardised (z-scored) inside the model loop regardless of raw units.


In [ ]:
scaling_factors = [1, 1, 10, 10, 10, 10, 1000, 1, 1, 1, 1, 1, 1, 10, 1, 1, 1]
new_cols_match = {}
new_cols_scoreline = {}
new_cols_possession = {}
for pi, factor in zip(performance_indicators, scaling_factors):
    for loc in ['home', 'away']:
        match_col = loc + '_' + pi
        if match_col in df_matchlevel.columns:
            new_cols_match[match_col + '_scaled'] = df_matchlevel[match_col] / factor
        sc_col = loc + '_' + pi + '_90'
        if sc_col in df_scorelinelevel.columns:
            new_cols_scoreline[sc_col + '_scaled'] = df_scorelinelevel[sc_col] / factor
        pos_col_90 = loc + '_' + pi + '_90'
        if pos_col_90 in df_possessionlevel.columns:
            new_cols_possession[pos_col_90 + '_scaled'] = df_possessionlevel[pos_col_90] / factor
        pos_col = loc + '_' + pi
        if pos_col in df_possessionlevel.columns:
            new_cols_possession[pos_col + '_scaled'] = df_possessionlevel[pos_col] / factor

df_matchlevel = df_matchlevel.assign(**new_cols_match)
df_scorelinelevel = df_scorelinelevel.assign(**new_cols_scoreline)
df_possessionlevel = df_possessionlevel.assign(**new_cols_possession)


/tmp/ipykernel_174969/1441014379.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_possessionlevel = df_possessionlevel.assign(**new_cols_possession)


In [ ]:
import statsmodels.api as sm
import scipy.stats as stats

from statsmodels.miscmodels.ordinal_model import OrderedModel

## Outcome Variables

| Level | Column | Values | Interpretation |
|-------|--------|--------|----------------|
| Season | `rank_category` | 0 (relegated) – 3 (Champions League) | Higher = better season outcome |
| Match | `match_outcome` | 0 (away win), 1 (draw), 2 (home win) | Home-team perspective |
| Scoreline | `next_goal` | 0 (away), 1 (none), 2 (home) | Which team scores next |
| Possession | `next_goal` | 0 (away), 1 (none), 2 (home) | Same as scoreline level |


In [ ]:
df_season['outcome_ordered'] = df_season['rank_category'].map(
    {'CL': 3, 'EL': 2, 'middle': 1, 'relegation': 0}).astype(int)

df_matchlevel['outcome_ordered'] = df_matchlevel['match_outcome'].map({
    'away_win': 0,
    'draw': 1,
    'home_win': 2
})

df_scorelinelevel['outcome_ordered'] = df_scorelinelevel['next_goal'].map({
    'away': 0,
    'none': 1,
    'home': 2
})

df_possessionlevel['outcome_ordered'] = df_possessionlevel['next_goal'].map({
    'away': 0,
    'none': 1,
    'home': 2
})


/tmp/ipykernel_174969/1295458238.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_possessionlevel['outcome_ordered'] = df_possessionlevel['next_goal'].map({


In [19]:
df_season.to_csv('../data/_df_seasonlevel.csv', index=False)
df_matchlevel.to_csv('../data/df_matchlevel.csv', index=False)
df_scorelinelevel.to_csv('../data/df_scorelinelevel.csv', index=False)
df_possessionlevel.to_csv('../data/df_possessionlevel.csv', index=False)